In [2]:
#import packages

import os
import re
import pandas as pd
import sklearn as sk
import numpy as np

In [3]:
#helper functions
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]

    #join these two dataframes
    df_temp = (
        df_pred_reduced
            .merge(df_gold_reduced, on=column_match, how='inner')   # keep only matching IDs
            .dropna(subset=[column_pred, column_gold])          # drop rows where values are NaN
            .reset_index(drop=True)                                 # tidy up the index
    )
    
    #ensure that the columns have data in the same type 
    df_temp[column_pred] = df_temp[column_pred].astype(int)
    df_temp[column_gold] = df_temp[column_gold].astype(int)

    #compute scores
    acc = round(sk.metrics.accuracy_score(df_temp[column_gold], df_temp[column_pred]), 2)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold], df_temp[column_pred]), 2)
    f1 = round(sk.metrics.f1_score(df_temp[column_gold], df_temp[column_pred]), 2)
    precision = round(sk.metrics.precision_score(df_temp[column_gold], df_temp[column_pred]), 2)
    recall = round(sk.metrics.recall_score(df_temp[column_gold], df_temp[column_pred]), 2)

    print("Accuracy for " + column_gold +  " is " + str(acc) + "%, and Cohen's kappa is " + str(k))

    return([acc, f1, precision, recall, k])

In [4]:
#read all model names
files = os.listdir("Predicted/")
models = list()
for file in files:
    if not re.search("scores", file) and not re.search(".DS_Store", file):
        models.append(file)


In [ ]:
#load data

#gold standard labels from Annotator 1 in round 2
df_gold = pd.read_csv('../Human/A1_r2.csv')

#create scores dataframe
scores_df = pd.DataFrame(index = ['Accuracy', 'F1', 'Precision', 'Recall', 'Kappa'],
                         columns = models)
for m in models:
    df = pd.read_csv('Predicted/' + m)

    scores_df.loc[:,m] = compute_scores(df_gold = df_gold, df_pred = df, column_match= "stories_id", column_gold= "relevance", column_pred= "relevance")

scores_df.to_csv('Scores.csv')

Accuracy for relevance is 0.92%, and Cohen's kappa is 0.81
Accuracy for relevance is 0.93%, and Cohen's kappa is 0.84
Accuracy for relevance is 0.92%, and Cohen's kappa is 0.8
Accuracy for relevance is 0.95%, and Cohen's kappa is 0.88
Accuracy for relevance is 0.95%, and Cohen's kappa is 0.89
